# Test ST-GCN++ badminton classifier (2 classes)

This notebook performs inference only. Select a T4 GPU, then run cells in order. The Conda installation intentionally restarts the runtime once.

In [ ]:
!nvidia-smi
!git clone --recurse-submodules https://github.com/dattt-cy/Badminton_AI.git /content/Badminton_AI
%cd /content/Badminton_AI

## Install Conda (run once)

This cell restarts the runtime. After Colab reconnects, continue with the next cell; do not run this cell again.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## Create the inference environment

Run this only after the runtime has reconnected. Installation can take several minutes.

In [ ]:
%cd /content/Badminton_AI
!CONDA_SOLVER=classic conda create -y -n pyskl_310 python=3.10 pip
!conda run -n pyskl_310 python -m pip install --no-cache-dir setuptools==69.5.1 numpy==1.23.5 scipy==1.9.3 pyyaml tqdm addict yapf==0.32.0 packaging termcolor pillow opencv-python-headless==4.7.0.72 fvcore==0.1.5.post20221221 filelock requests psutil py-cpuinfo matplotlib==3.7.5 pandas==1.5.3 seaborn==0.12.2
!conda run -n pyskl_310 python -m pip install --no-cache-dir torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113
!conda run -n pyskl_310 python -m pip install --no-cache-dir --no-deps mmcv-full==1.7.0 -f https://download.openmmlab.com/mmcv/dist/cu113/torch1.12.0/index.html
!conda run -n pyskl_310 python -m pip install --no-build-isolation --no-deps -e external/pyskl
!conda run -n pyskl_310 python -m pip install --no-deps -e .
!conda run -n pyskl_310 python -m pip install --no-deps ultralytics==8.3.0 ultralytics-thop==2.0.18
!conda run -n pyskl_310 python -m pip uninstall -y opencv-python
!conda run -n pyskl_310 python -m pip install --no-cache-dir --force-reinstall numpy==1.23.5 opencv-python-headless==4.7.0.72

In [ ]:
!conda run -n pyskl_310 python -c "import torch, mmcv, pyskl; from ultralytics import YOLO; print('torch', torch.__version__, 'mmcv', mmcv.__version__, 'cuda', torch.cuda.is_available(), 'ultralytics OK')"

## Upload the two-class checkpoint

Choose `best_top1_acc_epoch_13.pth` from your computer.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()
source = Path(next(iter(uploaded)))
checkpoint_path = Path('/content/Badminton_AI/models/checkpoints/action_recognition/best_top1_acc_2class.pth')
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
shutil.move(str(source), checkpoint_path)
print('Checkpoint:', checkpoint_path, checkpoint_path.stat().st_size)

## Upload one short video

Use a 1-4 second clip containing one complete action.

In [ ]:
from google.colab import files

uploaded = files.upload()
video_name = next(iter(uploaded))
print('Video:', video_name)

## Classify the action

Keep `--target single` for a one-player clip. Change it to `--target far` for the player on the far court in a match video.

In [ ]:
!PYTHONUNBUFFERED=1 conda run --no-capture-output -n pyskl_310 python -u scripts/inference/classify_action.py "$video_name" --checkpoint models/checkpoints/action_recognition/best_top1_acc_2class.pth --action-config configs/action_recognition/stgcnpp_badminton.py --target single --device cuda:0 --pose-output /content/test_pose.npz --json-output /content/test_result.json

In [ ]:
import json

with open('/content/test_result.json', encoding='utf-8') as result_file:
    result = json.load(result_file)
result